# 02 - Build Raw Layer

Este notebook constrói a camada Raw a partir dos arquivos originais armazenados na Landing Zone.

## Objetivos

- Ler os arquivos Parquet da landing.
- Tratar diferenças de schema físico entre arquivos mensais.
- Selecionar as colunas necessárias para o case.
- Aplicar casts técnicos mínimos.
- Adicionar metadados de origem e processamento.
- Persistir os dados como tabela Delta no Unity Catalog.

A camada Raw não aplica regras de negócio ou filtros de qualidade avançados. Essas regras serão tratadas na camada Silver/Trusted.

---

## 00. Criação da camada Bronze

#### 01. Libs Usadas

In [0]:
import os
import sys
import importlib

#### 02. Import do projeto

In [0]:
current_path = os.getcwd()

if current_path.endswith("/notebooks"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = current_path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for module_name in ["src.build_raw"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.build_raw as raw

print("Raw table:", raw.RAW_TABLE_NAME)
print("Landing base path:", raw.LANDING_BASE_PATH)

from src.build_raw import create_raw

Raw table: workspace.ifood_case.raw_yellow_taxi_trips
Landing base path: dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi


#### 03. Criar tabela Raw

In [0]:
YEAR = 2023
MONTHS = ["01", "02", "03", "04", "05"]

raw_df = create_raw(
    spark=spark,
    year=YEAR,
    months=MONTHS,
    mode="overwrite",
)

print(f"Raw records: {raw_df.count()}")

display(raw_df.limit(10))

Raw records: 16186386


VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime,source_year,source_month,source_file,raw_loaded_at
2,1.0,14.3,2023-01-01T00:32:10.000Z,2023-01-01T00:40:36.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,16.9,2023-01-01T00:55:08.000Z,2023-01-01T01:01:27.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,34.9,2023-01-01T00:25:04.000Z,2023-01-01T00:37:49.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
1,0.0,20.85,2023-01-01T00:03:48.000Z,2023-01-01T00:13:25.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,19.68,2023-01-01T00:10:29.000Z,2023-01-01T00:21:19.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,27.8,2023-01-01T00:50:34.000Z,2023-01-01T01:02:52.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,20.52,2023-01-01T00:09:22.000Z,2023-01-01T00:19:49.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,64.44,2023-01-01T00:27:12.000Z,2023-01-01T00:49:56.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,28.38,2023-01-01T00:21:44.000Z,2023-01-01T00:36:40.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z
2,1.0,19.9,2023-01-01T00:39:42.000Z,2023-01-01T00:50:36.000Z,2023,1,dbfs:/Volumes/workspace/ifood_case/case_files/landing/nyc_tlc/yellow_taxi/year=2023/month=01/yellow_tripdata_2023-01.parquet,2026-05-22T11:55:08.026Z


---

## 01. Validações da Raw

#### 01. Contagem por mês e Schema

In [0]:
display(
    spark.table("workspace.ifood_case.raw_yellow_taxi_trips")
    .groupBy("source_year", "source_month")
    .count()
    .orderBy("source_year", "source_month")
)

spark.table("workspace.ifood_case.raw_yellow_taxi_trips").printSchema()

source_year,source_month,count
2023,1,3066766
2023,2,2913955
2023,3,3403766
2023,4,3288250
2023,5,3513649


root
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- source_month: integer (nullable = true)
 |-- source_file: string (nullable = true)
 |-- raw_loaded_at: timestamp (nullable = true)



#### 02. Validação das colunas obrigatórias

In [0]:
required_columns = {
    "VendorID",
    "passenger_count",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
}

raw_columns = set(
    spark.table("workspace.ifood_case.raw_yellow_taxi_trips").columns
)

missing_columns = required_columns - raw_columns

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Todas as colunas solicitadas estão na camada Bronze.")

Todas as colunas solicitadas estão na camada Bronze.


#### 03. Confirmar tabela no catálogo

In [0]:
display(
    spark.sql("SHOW TABLES IN workspace.ifood_case")
)

database,tableName,isTemporary
ifood_case,gold_avg_passenger_count_may_by_hour,false
ifood_case,gold_avg_total_amount_by_month,false
ifood_case,gold_yellow_taxi_trips,false
ifood_case,raw_yellow_taxi_trips,false
ifood_case,silver_yellow_taxi_trips,false
